In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


In [2]:
#--------------------------------------------------------------------------------------------------------------------------
# config
#--------------------------------------------------------------------------------------------------------------------------

dataset_name = "Default"
output_name = "MSHT20N3LO-MC-chi2"
if_diag_only = False
index_file_name = "Index.csv"  # set to "Index.csv" or "full_index.csv" to force a specific file
merged_experiments = []#["E288", "E605", "E772"]


In [3]:
#--------------------------------------------------------------------------------------------------------------------------
# resolve paths
#--------------------------------------------------------------------------------------------------------------------------

def safe_folder_name(name, label):
    name = str(name).strip()
    if not name or name in {".", ".."} or "/" in name or "\\" in name:
        raise ValueError(f"{label} must be a single folder name, got {name!r}")
    return name


dataset_name = safe_folder_name(dataset_name, "dataset_name")
output_name = safe_folder_name(output_name, "output_name")

cwd = Path.cwd().resolve()
data_dir_candidates = [
    cwd,
    cwd / "Data",
    cwd.parent / "Data",
]
data_root = next(
    (
        path for path in data_dir_candidates
        if (path / dataset_name).exists() and (path / "Chi2_Matrix").exists()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError(
        f"Could not locate the Data directory from {cwd}. "
        "Run the notebook from TMD-Fits-Minimal or TMD-Fits-Minimal/Data."
    )

dy_root = (data_root / dataset_name / "Cutted" / "DY").resolve()
chi2_matrix_root = (data_root / "Chi2_Matrix").resolve()
chi2_after_dir = chi2_matrix_root

print(f"Data root: {data_root}")
print(f"DY source root: {dy_root}")
print(f"Chi2 matrix output: {chi2_after_dir}")


Data root: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data
DY source root: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Default\Cutted\DY
Chi2 matrix output: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix


## Helpers


In [4]:
def Build_Uncorrelated_Matrix(df, prefix="error_unc_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    if not cols:
        return np.zeros((len(df), len(df)), dtype=float)
    cols_squared_sum = df[cols].pow(2).sum(axis=1).to_numpy(dtype=float)
    return np.diag(cols_squared_sum)


def Build_Additive_Matrix(df, prefix="error_add_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    matrix_total = np.zeros((len(df), len(df)), dtype=float)
    for col_name in cols:
        col = df[col_name].to_numpy(dtype=float)
        matrix_total += np.outer(col, col)
    return matrix_total


def Build_Multiplicative_Matrix(df, prefix="error_mult_"):
    cols = [col for col in df.columns if col.startswith(prefix)]
    matrix_total = np.zeros((len(df), len(df)), dtype=float)
    xsec = df["xsec"].to_numpy(dtype=float)
    for col_name in cols:
        col = df[col_name].to_numpy(dtype=float) * xsec
        matrix_total += np.outer(col, col)
    return matrix_total


def Build_Covariance_Matrix(df):
    matrix_total = (
        Build_Uncorrelated_Matrix(df)
        + Build_Additive_Matrix(df)
        + Build_Multiplicative_Matrix(df)
    )
    if if_diag_only:
        matrix_total = np.diag(np.diag(matrix_total))
    return matrix_total


def resolve_data_file(root, relative_file):
    path = (root / relative_file).resolve()
    if root not in path.parents:
        raise ValueError(f"Resolved data file escaped DY root: {relative_file} -> {path}")
    return path


def find_index_path(folder, explicit_name=None):
    if explicit_name is not None:
        candidate = folder / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(f"Requested index file does not exist: {candidate}")
        return candidate

    candidates = [folder / "Index.csv", folder / "full_index.csv"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"No chi2 index file found under {folder}. Tried: {[str(p) for p in candidates]}"
    )


def block_key_for_file(file_name):
    experiment = Path(file_name).parts[0]
    if experiment in merged_experiments:
        return experiment
    return file_name


def combine_group_frames(frames):
    error_cols = sorted({col for df in frames for col in df.columns if col.startswith("error_")})
    aligned = []
    for df in frames:
        frame = df.copy()
        for col in error_cols:
            if col not in frame.columns:
                frame[col] = 0.0
        aligned.append(frame)
    return pd.concat(aligned, ignore_index=True, sort=False)


## Load Index


In [5]:
index_path = find_index_path(chi2_after_dir, explicit_name=index_file_name)
index_df = pd.read_csv(index_path)
required_cols = {"global_index", "file", "local_index"}
missing_cols = required_cols - set(index_df.columns)
if missing_cols:
    raise ValueError(f"Index file is missing required columns: {sorted(missing_cols)}")

index_df = index_df.sort_values("global_index").reset_index(drop=True)
expected_global = np.arange(len(index_df), dtype=int)
actual_global = index_df["global_index"].to_numpy(dtype=int)
if not np.array_equal(actual_global, expected_global):
    raise ValueError("Index global_index column must be contiguous and start at 0.")

index_df["block_key"] = index_df["file"].map(block_key_for_file)

print(f"Using index file: {index_path}")
print(f"Indexed points: {len(index_df)}")
print(f"Indexed files: {index_df['file'].nunique()}")
print(f"Merged experiments: {merged_experiments}")
print(f"Covariance blocks to build: {index_df['block_key'].nunique()}")


Using index file: C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix\Index.csv
Indexed points: 465
Indexed files: 57
Merged experiments: []
Covariance blocks to build: 57


## Build Experimental Blocks


In [6]:
exp_blocks = {}
group_offsets = {}

for block_key, block_group in index_df.groupby("block_key", sort=False):
    frames = []
    offsets = {}
    offset = 0

    for file_name in block_group["file"].drop_duplicates().tolist():
        file_rows = block_group[block_group["file"] == file_name]
        data_path = resolve_data_file(dy_root, file_name)
        df = pd.read_csv(data_path)

        local_indices = file_rows["local_index"].to_numpy(dtype=int)
        if np.any(local_indices < 0) or np.any(local_indices >= len(df)):
            raise IndexError(f"Index local_index out of bounds for {file_name}")
        if len(np.unique(local_indices)) != len(local_indices):
            raise ValueError(f"Duplicate local_index entries found for {file_name}")

        offsets[file_name] = offset
        frames.append(df)
        offset += len(df)

    combined_df = combine_group_frames(frames)
    exp_blocks[block_key] = Build_Covariance_Matrix(combined_df)
    group_offsets[block_key] = offsets

print(f"Built {len(exp_blocks)} experimental covariance blocks.")


Built 57 experimental covariance blocks.


## Assemble Full Matrix


In [7]:
n_points = len(index_df)
Exp_Matrix = np.zeros((n_points, n_points), dtype=float)

for block_key, block_group in index_df.groupby("block_key", sort=False):
    block = exp_blocks[block_key]
    offsets = group_offsets[block_key]
    positions = block_group["global_index"].to_numpy(dtype=int)
    combined_indices = np.array(
        [offsets[file_name] + int(local_index) for file_name, local_index in zip(block_group["file"], block_group["local_index"])],
        dtype=int,
    )
    Exp_Matrix[np.ix_(positions, positions)] = block[np.ix_(combined_indices, combined_indices)]

print(f"Experimental chi2 covariance shape: {Exp_Matrix.shape}")


Experimental chi2 covariance shape: (465, 465)


## Write Output


In [8]:
chi2_after_dir.mkdir(parents=True, exist_ok=True)

exp_destination = chi2_after_dir / "Exp_cov.csv"
if exp_destination.exists():
    raise FileExistsError(f"{exp_destination} already exists")

pd.DataFrame(Exp_Matrix).to_csv(exp_destination, index=False)
print(f"{exp_destination} generated")


C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix\Exp_cov.csv generated


## Total Covariance Inverse


In [9]:
pdf_cov_path = chi2_after_dir / "PDF_cov.csv"
if not pdf_cov_path.exists():
    raise FileNotFoundError(f"Missing PDF covariance file: {pdf_cov_path}")

PDF_Matrix = pd.read_csv(pdf_cov_path).to_numpy(dtype=float)
if PDF_Matrix.shape != Exp_Matrix.shape:
    raise ValueError(
        f"PDF covariance shape {PDF_Matrix.shape} does not match experimental covariance shape {Exp_Matrix.shape}"
    )

Total_Matrix = Exp_Matrix + PDF_Matrix
Total_Matrix = 0.5 * (Total_Matrix + Total_Matrix.T)
Total_Inverse = np.linalg.inv(Total_Matrix)

inverse_destination = chi2_after_dir / "Total_inverse.csv"
if inverse_destination.exists():
    raise FileExistsError(f"{inverse_destination} already exists")

pd.DataFrame(Total_Inverse).to_csv(inverse_destination, index=False)
print(f"{inverse_destination} generated")
print(f"Total covariance shape: {Total_Matrix.shape}")


C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Data\Chi2_Matrix\Total_inverse.csv generated
Total covariance shape: (465, 465)
